# 01 — Ingest: Download 311 FeatureServer Data

Fetches all records for a given year from the Baltimore Open Data ArcGIS FeatureServer
and saves them to `data/raw/requests_{YEAR}.parquet`.

**Run locally** — the FeatureServer IP-restricts access from some cloud environments.
Expect ~150–200k records per year; the full fetch takes 5–15 minutes.

**Output:** `data/raw/requests_{YEAR}.parquet`

In [ ]:
import sys
from pathlib import Path

# Make src/balt311 importable from notebooks/
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import pandas as pd
from balt311.ingest import fetch_year

YEAR = 2024          # <── change to target year
RAW_DIR = Path('..') / 'data' / 'raw'
RAW_DIR.mkdir(exist_ok=True)
OUT = RAW_DIR / f'requests_{YEAR}.parquet'
print(f'Output: {OUT}')

In [ ]:
print(f'Fetching {YEAR} 311 data ...')
records = fetch_year(YEAR)
print(f'\nTotal records fetched: {len(records):,}')

In [ ]:
df = pd.DataFrame(records)
df.to_parquet(OUT, index=False)
print(f'Saved {len(df):,} rows → {OUT}')
print(f'\nColumns ({len(df.columns)}):')
print(df.dtypes.to_string())

In [ ]:
# Quick sanity checks
from collections import Counter
import datetime

print(f'Rows:     {len(df):,}')
print(f'Columns:  {len(df.columns)}')
print(f'Null SRRecordID: {df["SRRecordID"].isna().sum()}')
print(f'Duplicate SRRecordID: {df["SRRecordID"].duplicated().sum()}')
print(f'Null Latitude:  {df["Latitude"].isna().sum()} ({100*df["Latitude"].isna().mean():.1f}%)')
print(f'Null Longitude: {df["Longitude"].isna().sum()} ({100*df["Longitude"].isna().mean():.1f}%)')